In [10]:
# Generate chimeric RNA FASTA and read depth files to use when generating
# simulated HPV integation RNA-seq reads via polyester

import sys
import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq
import random
import math

# Load Data
sim_sites = pd.read_csv("sim_sites.txt", sep="\t")
hpv_splicings = pd.read_csv("hpv16_splicings.txt", sep="\t")
exons = pd.read_csv("geneExons.csv")  # columns: gene, chr, exon, start, stop

# Load FASTAs
hpv_ref = str(SeqIO.read("HPV16REF.fa", "fasta").seq)
hg19_dict = SeqIO.to_dict(SeqIO.parse("hg19.fa", "fasta"))

# Filter detectable HPV isoforms
detectable_hpv = hpv_splicings[hpv_splicings["detectable"] == "T"]

def get_hpv_spliced_seq(isoform_letter):
    """Extracts and concatenates spliced HPV exons for a given isoform letter."""
    iso_exons = detectable_hpv[detectable_hpv["splicing"] == isoform_letter].sort_values("exon")
    seq_parts = []
    for _, row in iso_exons.iterrows():
        # 1-based start/stop coordinates from table converted to 0-based indexing
        seq_parts.append(hpv_ref[row["start"] - 1 : row["stop"]])
    return seq_parts

def get_human_exon_seq(chrom, start, stop, strand):
    """Extracts human genomic exon sequence and handles strand orientation."""
    raw_seq = str(hg19_dict[chrom].seq[start - 1 : stop])
    if strand == "-":
        return str(Seq(raw_seq).reverse_complement())
    return raw_seq

output_fastas = {}
count_matrix = {}

# Iterate through simulation sites
for idx, row in sim_sites.iterrows():
    cid = row["Construct"]
    gene = row["Name"]
    depth = row["Read Depth"]
    strand = row["Strand"]
    insert_site = row["Insert site"]

    # Skip silenced intergenic constructs (Construct 5)
    if cid == 5 or depth == 0:
        continue

    # Get gene exon annotations sorted in transcriptional order (5' -> 3')
    gene_exons = exons[exons["gene"] == gene].copy()
    chrom = gene_exons["chr"].iloc[0]
    
    if strand == "+":
        gene_exons = gene_exons.sort_values("exon", ascending=True)
    else:
        gene_exons = gene_exons.sort_values("exon", ascending=True) # Exon numbers 1..N correspond to 5'->3'

    intron_num = int(insert_site)
    
    # Cycle through detectable HPV isoforms
    spliceLetters = detectable_hpv['splicing'].unique().tolist()
    lenLetters = len(spliceLetters)
    isoform_choice = spliceLetters[idx % lenLetters]
    hpv_seq = get_hpv_spliced_seq(isoform_choice)
    
    chimeric_seq = ""

    # Construct 1: Canonical insertion (Viral 5' -> Host 3' downstream exons)
    if cid == 1:
        # Downstream exons start after intron_num (Exon intron_num + 1 to end)
        downstream_exons = gene_exons[gene_exons["exon"] > intron_num]
        host_seq = "".join([get_human_exon_seq(chrom, r["start"], r["stop"], strand) for _, r in downstream_exons.iterrows()])
        
        # Randomly choose an exon to stop on:
        stopExon = random.randint(1, len(hpv_seq)-1)
        tx_name = f"Construct_{cid}_HPV16_{isoform_choice}_toExon{stopExon}_to_{gene}_intron{intron_num}"
        hpv_seq = "".join(hpv_seq[:stopExon])
        chimeric_seq = hpv_seq + host_seq

    # Construct 2: Concomitant host deletion (Skip next 2 exons)
    elif cid == 2:
        downstream_exons = gene_exons[gene_exons["exon"] > (intron_num + 2)]
        host_seq = "".join([get_human_exon_seq(chrom, r["start"], r["stop"], strand) for _, r in downstream_exons.iterrows()])
        
        # Randomly choose an exon to stop on:
        stopExon = random.randint(1, len(hpv_seq)-1)
        tx_name = f"Construct_{cid}_HPV16_{isoform_choice}_toExon{stopExon}_to_{gene}_intron{intron_num}"
        hpv_seq = "".join(hpv_seq[:stopExon])
        chimeric_seq = hpv_seq + host_seq

    # Construct 3: Concatemer (Tandem full HPV + spliced chimera)
    elif cid == 3:
        downstream_exons = gene_exons[gene_exons["exon"] > intron_num]
        host_seq = "".join([get_human_exon_seq(chrom, r["start"], r["stop"], strand) for _, r in downstream_exons.iterrows()])
        
        # Randomly choose an exon to stop on:
        stopExon = random.randint(1, len(hpv_seq)-1)
        tx_name = f"Construct_{cid}_HPV16_{isoform_choice}_toExon{stopExon}_to_{gene}_intron{intron_num}"
        hpv_seq = "".join(hpv_seq[:stopExon])

        # Tandem repeat: Full HPV genome + Spliced Chimera
        chimeric_seq = hpv_ref + hpv_seq + host_seq

    # Construct 4: Inverted / Antisense (Host 5' upstream exons -> Reverse Complement HPV)
    elif cid == 4:
        upstream_exons = gene_exons[gene_exons["exon"] <= intron_num]
        host_seq = "".join([get_human_exon_seq(chrom, r["start"], r["stop"], strand) for _, r in upstream_exons.iterrows()])
                
        #Randomly choose an exon to stop on, leaving out the final exon w/polyA site:
        stopExon = random.randint(1, len(hpv_seq)-1)
        tx_name = f"Construct_{cid}_HPV16_{isoform_choice}_toExon{stopExon}_to_{gene}_intron{intron_num}"
        hpv_seq = "".join(hpv_seq[:stopExon])
        hpv_revcomp = str(Seq(hpv_seq).reverse_complement())
        chimeric_seq = host_seq + hpv_revcomp

    output_fastas[tx_name] = chimeric_seq
    count_matrix[tx_name] = depth

# Write output files
with open("chimeric_transcripts.fasta", "w") as fa_out:
    for header, seq in output_fastas.items():
        fa_out.write(f">{header}\n{seq}\n")

with open("reads_per_transcript.txt", "w") as txt_out:
    txt_out.write("transcript\tcount\n")
    for header, count in count_matrix.items():
        txt_out.write(f"{header}\t{count}\n")

print("Generated chimeric_transcripts.fasta and reads_per_transcript.txt successfully.")

Generated chimeric_transcripts.fasta and reads_per_transcript.txt successfully.
